In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
import numpy as np

from src.preprocessing import preprocess_data
from src.features import add_structural_features
from sklearn.model_selection import train_test_split

In [2]:
df = pd.read_csv("../data/train.csv")

df = preprocess_data(df)
df = add_structural_features(df)

label_map = {
    "A": 0,
    "B": 1,
    "tie": 2
}

df["target"] = df["label"].map(label_map)

In [3]:
train_df, temp_df = train_test_split(
    df,
    test_size=0.2,
    random_state=714,
    stratify=df["target"]
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.5,
    random_state=714,
    stratify=temp_df["target"]
)

print(train_df.shape)
print(val_df.shape)
print(test_df.shape)

(45981, 25)
(5748, 25)
(5748, 25)


# tokenize

In [10]:
import re

def tokenize(text):
    return re.findall(
        r"\b\w+\b",
        text.lower()
    )

In [11]:
def build_candidate_tokens(prompt_turns, response_turns):
    tokens = []

    for prompt, response in zip(prompt_turns, response_turns):

        tokens.append("<USER>")

        if isinstance(prompt, str):
            tokens.extend(tokenize(prompt))

        tokens.append("<ASSISTANT>")

        if isinstance(response, str):
            tokens.extend(tokenize(response))
        else:
            tokens.append("<EMPTY>")

    return tokens

In [12]:
train_df["candidate_a_tokens"] = train_df.apply(
    lambda row: build_candidate_tokens(
        row["prompt_list"],
        row["response_a_list"]
    ),
    axis=1
)

train_df["candidate_b_tokens"] = train_df.apply(
    lambda row: build_candidate_tokens(
        row["prompt_list"],
        row["response_b_list"]
    ),
    axis=1
)

In [13]:
sample = train_df[
    train_df["prompt_turns"] > 1
].iloc[0]

print(sample["candidate_a_tokens"][:100])

['<USER>', 'o', 'que', 'é', 'sincerismo', '<ASSISTANT>', 'sincerismo', 'é', 'um', 'termo', 'utilizado', 'em', 'filosofia', 'e', 'ética', 'para', 'designar', 'a', 'ideologia', 'que', 'defende', 'que', 'a', 'verdade', 'e', 'a', 'sinceridade', 'devem', 'ser', 'priorizadas', 'em', 'todas', 'as', 'interações', 'humanas', 'especialmente', 'em', 'relação', 'às', 'relações', 'interpessoais', 'o', 'sincerismo', 'argumenta', 'que', 'a', 'falta', 'de', 'sinceridade', 'e', 'a', 'mentira', 'são', 'danosas', 'às', 'relações', 'interpessoais', 'pois', 'erode', 'a', 'confiança', 'e', 'a', 'trustworthiness', 'entre', 'as', 'pessoas', 'por', 'isso', 'o', 'sincerismo', 'busca', 'promover', 'a', 'honestidade', 'a', 'transparência', 'e', 'a', 'sinceridade', 'em', 'todas', 'as', 'interações', 'humanas', 'desde', 'as', 'relações', 'pessoais', 'até', 'as', 'relações', 'políticas', 'e', 'sociais', 'o', 'sincerismo', 'também', 'pode', 'ser']


In [14]:
train_df["candidate_a_len"] = (
    train_df["candidate_a_tokens"].apply(len)
)

train_df["candidate_b_len"] = (
    train_df["candidate_b_tokens"].apply(len)
)

In [15]:
train_df[
    [
        "candidate_a_len",
        "candidate_b_len"
    ]
].describe(
    percentiles=[0.5, 0.75, 0.9, 0.95, 0.99]
)

,candidate_a_len,candidate_b_len
count,45981.000000,45981.000000
mean,276.409669,277.573519
std,301.374238,303.321811
min,4.000000,4.000000
50%,214.000000,215.000000
75%,350.000000,350.000000
90%,538.000000,542.000000
95%,740.000000,739.000000
99%,1500.000000,1508.200000
max,11106.000000,9209.000000
